# M06C: Solutions

Complete solution for the M06C Capstone exercise.

**Exercise:** Add a refund tool (database write operation).

---

## 🔧 Step 1: Setup

In [ ]:
import os
import json
import sqlite3
import requests
from pathlib import Path
from dotenv import load_dotenv

import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"

INSTRUCTIONS = (
    "You are a helpful assistant. "
    "Use the supplied tools to answer user questions."
)


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🏗️ Prerequisites: Re-creating the M06C Environment

This cell recreates the database, functions, tools, and `IntegrationAssistant` from M06C so the solutions can run standalone.

In [ ]:
# --------------------------------------------------------------
# Database Setup
# --------------------------------------------------------------

def setup_database():
    """Create sample database."""
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            name TEXT,
            email TEXT,
            plan TEXT
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE orders (
            id INTEGER PRIMARY KEY,
            user_id INTEGER,
            product TEXT,
            amount REAL,
            status TEXT
        )
    ''')
    
    cursor.executemany('INSERT INTO users VALUES (?, ?, ?, ?)', [
        (1, 'Alice', 'alice@example.com', 'premium'),
        (2, 'Bob', 'bob@example.com', 'free'),
        (3, 'Carol', 'carol@example.com', 'premium')
    ])
    
    cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?)', [
        (101, 1, 'Widget', 29.99, 'shipped'),
        (102, 2, 'Gadget', 49.99, 'pending'),
        (103, 1, 'Tool', 19.99, 'delivered')
    ])
    
    conn.commit()
    return conn

db = setup_database()

# --------------------------------------------------------------
# Weather Function
# --------------------------------------------------------------

def get_coordinates(city):
    """Get lat/lon from Open-Meteo geocoding API."""
    try:
        # Open-Meteo: free weather API, no key required
        response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=5
        )
        response.raise_for_status()
        results = response.json().get("results", [])
        if results:
            return {"lat": results[0]["latitude"], "lon": results[0]["longitude"]}
        return None
    except Exception:
        return None

def get_live_weather(location):
    """Get real weather data from API."""
    try:
        location = location.strip()
        coords = get_coordinates(location)
        if not coords:
            return {"error": "City not found"}
        
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": coords["lat"],
            "longitude": coords["lon"],
            "current": "temperature_2m,wind_speed_10m,weather_code"
        }
        
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()
        data = response.json()
        current = data.get("current", {})
        
        return {
            "location": location,
            "temperature_c": current.get("temperature_2m"),
            "windspeed_kmh": current.get("wind_speed_10m"),
            "weather_code": current.get("weather_code")
        }
    except requests.Timeout:
        return {"error": "API timeout - try again"}
    except requests.RequestException as e:
        return {"error": f"API error: {str(e)}"}
    except Exception as e:
        return {"error": f"Error: {str(e)}"}

# --------------------------------------------------------------
# Database Functions
# --------------------------------------------------------------

def get_user_by_id(user_id):
    """Get user from database."""
    try:
        user_id = int(user_id)
        cursor = db.cursor()
        cursor.execute('SELECT * FROM users WHERE id = ?', (user_id,))
        row = cursor.fetchone()
        if row:
            return {"id": row[0], "name": row[1], "email": row[2], "plan": row[3]}
        return {"error": "User not found"}
    except Exception as e:
        return {"error": str(e)}

def get_user_orders(user_id):
    """Get all orders for user."""
    try:
        user_id = int(user_id)
        cursor = db.cursor()
        cursor.execute('SELECT * FROM orders WHERE user_id = ?', (user_id,))
        rows = cursor.fetchall()
        orders = [
            {"order_id": row[0], "product": row[2], "amount": row[3], "status": row[4]}
            for row in rows
        ]
        return {"orders": orders}
    except Exception as e:
        return {"error": str(e)}

def update_order_status(order_id, new_status):
    """Update order status in database."""
    try:
        order_id = int(order_id)
        cursor = db.cursor()
        cursor.execute('UPDATE orders SET status = ? WHERE id = ?', (new_status, order_id))
        db.commit()
        if cursor.rowcount > 0:
            return {"success": True, "message": f"Order {order_id} updated to '{new_status}'"}
        return {"error": f"Order {order_id} not found"}
    except Exception as e:
        return {"error": str(e)}

# --------------------------------------------------------------
# Tool Schemas
# --------------------------------------------------------------

weather_tools = [
    {
        "type": "function",
        "name": "get_live_weather",
        "description": "Get current weather for a city using live API data",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, e.g. San Francisco"
                }
            },
            "required": ["location"]
        }
    }
]

db_tools = [
    {
        "type": "function",
        "name": "get_user_by_id",
        "description": "Get user information from database by ID",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "integer", "description": "User ID"}
            },
            "required": ["user_id"]
        }
    },
    {
        "type": "function",
        "name": "get_user_orders",
        "description": "Get all orders for a user",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {"type": "integer", "description": "User ID"}
            },
            "required": ["user_id"]
        }
    },
    {
        "type": "function",
        "name": "update_order_status",
        "description": "Update the status of an order",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "integer", "description": "Order ID"},
                "new_status": {"type": "string", "description": "New status (e.g. shipped, delivered, refunded)"}
            },
            "required": ["order_id", "new_status"]
        }
    }
]

# --------------------------------------------------------------
# IntegrationAssistant
# --------------------------------------------------------------

class IntegrationAssistant:
    """Production assistant with multiple integrations."""
    
    def __init__(self, client, model, instructions=None):
        self.client = client
        self.model = model
        self.instructions = instructions
        self.call_log = []
        self.error_count = 0
        
        self.tools = weather_tools + db_tools
        
        self.functions = {
            "get_live_weather": get_live_weather,
            "get_user_by_id": get_user_by_id,
            "get_user_orders": get_user_orders,
            "update_order_status": update_order_status
        }
    
    def chat(self, user_message, max_turns=5):
        """Chat with comprehensive error handling."""
        conversation_items = [{"role": "user", "content": user_message}]
        
        for _ in range(max_turns):
            try:
                response = self.client.responses.create(
                    model=self.model,
                    input=conversation_items,
                    tools=self.tools,
                    instructions=self.instructions
                )
                
                tool_items = [
                    item for item in response.output
                    if item.type == "function_call"
                ]
                
                if not tool_items:
                    return response.output_text
                
                for item in tool_items:
                    function_name = item.name
                    function_args = json.loads(item.arguments)
                    
                    self.call_log.append({
                        "function": function_name,
                        "args": function_args
                    })
                    
                    conversation_items.append({
                        "type": "function_call",
                        "call_id": item.call_id,
                        "name": function_name,
                        "arguments": item.arguments
                    })
                    
                    try:
                        result = self.functions[function_name](**function_args)
                        if isinstance(result, dict) and "error" in result:
                            self.error_count += 1
                    except Exception as e:
                        result = {"error": str(e)}
                        self.error_count += 1
                    
                    conversation_items.append({
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": json.dumps(result)
                    })
            
            except Exception as e:
                self.error_count += 1
                return f"Error: {str(e)}"
        
        return "Error: too many tool-call rounds."
    
    def stats(self):
        """Get statistics."""
        return {
            "total_calls": len(self.call_log),
            "errors": self.error_count,
            "functions_used": sorted(set(c["function"] for c in self.call_log))
        }

assistant = IntegrationAssistant(
    client=client,
    model=MODEL,
    instructions=INSTRUCTIONS
)

print("✅ M06C Environment Recreated")

---

## ✨ Solution: Refund Tool

Add a `process_refund` function that checks if an order exists, then updates its status to "refunded".

In [ ]:
def process_refund(order_id):
    """Process refund for an order."""
    try:
        order_id = int(order_id)
        cursor = db.cursor()
        
        # Check if order exists
        cursor.execute('SELECT status FROM orders WHERE id = ?', (order_id,))
        row = cursor.fetchone()
        if not row:
            return {"error": f"Order {order_id} not found"}
        
        # Update status
        cursor.execute('UPDATE orders SET status = ? WHERE id = ?', ('refunded', order_id))
        db.commit()
        
        if cursor.rowcount > 0:
            return {"success": True, "message": f"Order {order_id} refunded"}
        return {"error": "Refund failed"}
    except Exception as e:
        return {"error": str(e)}

# Tool schema
refund_tool = {
    "type": "function",
    "name": "process_refund",
    "description": "Process a refund for an order",
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {"type": "integer", "description": "The order ID to refund"}
        },
        "required": ["order_id"]
    }
}

# Extend the assistant's registries
all_tools = assistant.tools + [refund_tool]
all_functions = {**assistant.functions, "process_refund": process_refund}

# Create extended assistant
refund_assistant = IntegrationAssistant(
    client=client,
    model=MODEL,
    instructions=INSTRUCTIONS
)
refund_assistant.tools = all_tools
refund_assistant.functions = all_functions

# Test
print("🤖 REFUND TOOL TEST")
print("="*60)
print(f"User: Refund order 101")
print(f"Agent: {refund_assistant.chat('Refund order 101')}")

# Verify DB state
print("\n🔍 DB Verification:")
cursor = db.cursor()
cursor.execute('SELECT id, status FROM orders WHERE user_id = 1')
for row in cursor.fetchall():
    print(f"  Order {row[0]}: {row[1]}")

---

## 🎯 Key Takeaways

**Extending an Agent:**
Define the function, create the schema, and add both to the registries. Use `tools + [new_tool]` and `{**funcs, "name": func}` to extend without modifying originals.

**Database Write Operations:**
Always check if the record exists first (SELECT), then update (UPDATE), then verify (`cursor.rowcount`), and commit (`db.commit()`).